# Comparação de Modelos - Previsão de Churn

**FGV - Modelagem Preditiva**

**Grupo:** João Victor Fortunato Teixeira Mundim, Martinho Aloisio Lenz, Rodrigo Pimentel, Bruno Barradas

Este notebook compara os 4 modelos de classificação para previsão de Churn:
1. Árvore de Decisão (ID3/Entropy)
2. Regressão Logística
3. Random Forest
4. Gradient Boosting

In [ ]:
####################################
# COMPARAÇÃO DE MODELOS - CHURN    #
####################################

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

print("Bibliotecas importadas com sucesso.")

In [ ]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================
CAMINHO_BASE = '../dados/'
ARQUIVO_TREINO = CAMINHO_BASE + 'Churn_de_clientes_de_Internet_-_dados_de_treinamento_8122.xlsx'
ARQUIVO_CLASSIFICAR = CAMINHO_BASE + 'Clientes_reais_para_serem_classificados_c7fc.xlsx'

random_state = 42
test_size = 0.2

In [ ]:
# ============================================================
# CARREGAMENTO E PRÉ-PROCESSAMENTO
# ============================================================
print("=== Carregando e preparando dados ===")

df = pd.read_excel(ARQUIVO_TREINO)
print(f"Dataset: {df.shape[0]} registros, {df.shape[1]} colunas")
print(f"Proporção de Churn: {df['Classe'].mean()*100:.1f}%")

# Preparar dados
X = df.drop(columns=['ID_Cliente', 'Classe']).copy()
y = df['Classe'].copy()

colunas_cat = X.select_dtypes(include=['object']).columns.tolist()
colunas_num = X.select_dtypes(include=['number']).columns.tolist()

# Valores ausentes
medianas = {}
for col in colunas_num:
    if X[col].isnull().sum() > 0:
        medianas[col] = X[col].median()
        X[col] = X[col].fillna(medianas[col])

for col in colunas_cat:
    X[col] = X[col].fillna('Desconhecido')

# Label Encoding para modelos baseados em árvore
X_tree = X.copy()
encoders = {}
for col in colunas_cat:
    le = LabelEncoder()
    X_tree[col] = le.fit_transform(X_tree[col].astype(str))
    encoders[col] = le

# One-Hot + Scaling para Regressão Logística
X_logistic = pd.get_dummies(X, columns=colunas_cat, drop_first=True)
scaler = StandardScaler()
X_logistic_scaled = pd.DataFrame(scaler.fit_transform(X_logistic), columns=X_logistic.columns)

print("\nDados preparados com sucesso.")

In [ ]:
# ============================================================
# DIVISÃO TREINO/TESTE
# ============================================================
X_tree_train, X_tree_test, y_train, y_test = train_test_split(
    X_tree, y, test_size=test_size, random_state=random_state, stratify=y
)

X_log_train, X_log_test, _, _ = train_test_split(
    X_logistic_scaled, y, test_size=test_size, random_state=random_state, stratify=y
)

print(f"Treino: {len(y_train)} amostras")
print(f"Teste: {len(y_test)} amostras")

In [ ]:
# ============================================================
# DEFINIÇÃO E TREINAMENTO DOS MODELOS
# ============================================================
print("=== Treinando os 4 modelos ===")

modelos = {
    'Árvore de Decisão': {
        'modelo': DecisionTreeClassifier(
            criterion='entropy', max_depth=5, min_samples_split=10,
            min_samples_leaf=5, random_state=random_state
        ),
        'X_train': X_tree_train, 'X_test': X_tree_test
    },
    'Regressão Logística': {
        'modelo': LogisticRegression(
            penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=random_state
        ),
        'X_train': X_log_train, 'X_test': X_log_test
    },
    'Random Forest': {
        'modelo': RandomForestClassifier(
            n_estimators=100, criterion='entropy', max_features='sqrt',
            bootstrap=True, random_state=random_state, n_jobs=-1
        ),
        'X_train': X_tree_train, 'X_test': X_tree_test
    },
    'Gradient Boosting': {
        'modelo': GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=3,
            subsample=0.8, random_state=random_state
        ),
        'X_train': X_tree_train, 'X_test': X_tree_test
    }
}

# Treinar modelos
for nome, config in modelos.items():
    print(f"  Treinando {nome}...")
    config['modelo'].fit(config['X_train'], y_train)

print("\nTodos os modelos treinados!")

In [ ]:
# ============================================================
# AVALIAÇÃO DOS MODELOS
# ============================================================
print("=== Avaliação dos Modelos ===")

resultados = []

for nome, config in modelos.items():
    modelo = config['modelo']
    X_train_m = config['X_train']
    X_test_m = config['X_test']
    
    # Predições
    y_pred_train = modelo.predict(X_train_m)
    y_pred_test = modelo.predict(X_test_m)
    y_proba_test = modelo.predict_proba(X_test_m)[:, 1]
    
    # Métricas
    acc_train = accuracy_score(y_train, y_pred_train)
    acc_test = accuracy_score(y_test, y_pred_test)
    auc = roc_auc_score(y_test, y_proba_test)
    precision = precision_score(y_test, y_pred_test)
    recall = recall_score(y_test, y_pred_test)
    f1 = f1_score(y_test, y_pred_test)
    
    # Validação cruzada
    X_full = X_tree if nome != 'Regressão Logística' else X_logistic_scaled
    cv_scores = cross_val_score(modelo, X_full, y, cv=5, scoring='accuracy')
    
    # Guardar resultados
    config['y_pred_test'] = y_pred_test
    config['y_proba_test'] = y_proba_test
    
    resultados.append({
        'Modelo': nome,
        'Acurácia Treino': acc_train,
        'Acurácia Teste': acc_test,
        'AUC-ROC': auc,
        'Precisão': precision,
        'Recall': recall,
        'F1-Score': f1,
        'CV Média': cv_scores.mean(),
        'CV Std': cv_scores.std()
    })

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("COMPARAÇÃO DE DESEMPENHO")
print("="*80)
display(df_resultados.style.format({
    'Acurácia Treino': '{:.2%}',
    'Acurácia Teste': '{:.2%}',
    'AUC-ROC': '{:.4f}',
    'Precisão': '{:.2%}',
    'Recall': '{:.2%}',
    'F1-Score': '{:.2%}',
    'CV Média': '{:.2%}',
    'CV Std': '{:.2%}'
}))

In [ ]:
# ============================================================
# VISUALIZAÇÃO COMPARATIVA
# ============================================================
print("=== Visualizações Comparativas ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Comparação de Métricas
metricas = ['Acurácia Teste', 'AUC-ROC', 'Precisão', 'Recall']
x = np.arange(len(df_resultados))
width = 0.2

for i, metrica in enumerate(metricas):
    axes[0, 0].bar(x + i*width, df_resultados[metrica], width, label=metrica)

axes[0, 0].set_xlabel('Modelo')
axes[0, 0].set_ylabel('Valor')
axes[0, 0].set_title('Comparação de Métricas por Modelo')
axes[0, 0].set_xticks(x + width * 1.5)
axes[0, 0].set_xticklabels(df_resultados['Modelo'], rotation=45, ha='right')
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1)

# 2. Curvas ROC
colors = ['blue', 'purple', 'orange', 'green']
for i, (nome, config) in enumerate(modelos.items()):
    fpr, tpr, _ = roc_curve(y_test, config['y_proba_test'])
    auc = roc_auc_score(y_test, config['y_proba_test'])
    axes[0, 1].plot(fpr, tpr, color=colors[i], lw=2, label=f'{nome} (AUC={auc:.3f})')

axes[0, 1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[0, 1].set_xlabel('Taxa de Falso Positivo')
axes[0, 1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[0, 1].set_title('Curvas ROC Comparativas')
axes[0, 1].legend(loc='lower right')
axes[0, 1].grid(True, alpha=0.3)

# 3. Acurácia Treino vs Teste (Overfitting)
x = np.arange(len(df_resultados))
width = 0.35
axes[1, 0].bar(x - width/2, df_resultados['Acurácia Treino'], width, label='Treino', color='steelblue')
axes[1, 0].bar(x + width/2, df_resultados['Acurácia Teste'], width, label='Teste', color='coral')
axes[1, 0].set_xlabel('Modelo')
axes[1, 0].set_ylabel('Acurácia')
axes[1, 0].set_title('Acurácia Treino vs Teste (Análise de Overfitting)')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(df_resultados['Modelo'], rotation=45, ha='right')
axes[1, 0].legend()
axes[1, 0].set_ylim(0.5, 1.05)

# 4. Validação Cruzada
axes[1, 1].bar(df_resultados['Modelo'], df_resultados['CV Média'], 
               yerr=df_resultados['CV Std'], capsize=5, color='teal', alpha=0.7)
axes[1, 1].set_xlabel('Modelo')
axes[1, 1].set_ylabel('Acurácia Média')
axes[1, 1].set_title('Validação Cruzada (5-fold)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MATRIZES DE CONFUSÃO
# ============================================================
print("=== Matrizes de Confusão ===")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
cmaps = ['Blues', 'Purples', 'Oranges', 'Greens']

for i, (nome, config) in enumerate(modelos.items()):
    ax = axes[i//2, i%2]
    cm = confusion_matrix(y_test, config['y_pred_test'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmaps[i], ax=ax,
                xticklabels=['Não Churn', 'Churn'],
                yticklabels=['Não Churn', 'Churn'])
    ax.set_title(f'Matriz de Confusão - {nome}')
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASSIFICAÇÃO DOS CLIENTES REAIS
# ============================================================
print("=== Classificação dos Clientes Reais ===")

try:
    df_classificar = pd.read_excel(ARQUIVO_CLASSIFICAR)
    print(f"Clientes para classificar: {len(df_classificar)}")
    
    ids_clientes = df_classificar['ID_Cliente'].copy() if 'ID_Cliente' in df_classificar.columns else range(1, len(df_classificar)+1)
    
    # Preparar dados para modelos tree-based
    X_novo_tree = df_classificar.drop(columns=['ID_Cliente', 'Classe'], errors='ignore').copy()
    for col in colunas_num:
        if col in X_novo_tree.columns and col in medianas:
            X_novo_tree[col] = X_novo_tree[col].fillna(medianas[col])
    for col in colunas_cat:
        if col in X_novo_tree.columns:
            X_novo_tree[col] = X_novo_tree[col].fillna('Desconhecido')
            le = encoders[col]
            X_novo_tree[col] = X_novo_tree[col].astype(str).apply(lambda v: v if v in le.classes_ else le.classes_[0])
            X_novo_tree[col] = le.transform(X_novo_tree[col])
    X_novo_tree = X_novo_tree[X_tree.columns]
    
    # Preparar dados para Regressão Logística
    X_novo_log = df_classificar.drop(columns=['ID_Cliente', 'Classe'], errors='ignore').copy()
    for col in colunas_num:
        if col in X_novo_log.columns and col in medianas:
            X_novo_log[col] = X_novo_log[col].fillna(medianas[col])
    for col in colunas_cat:
        if col in X_novo_log.columns:
            X_novo_log[col] = X_novo_log[col].fillna('Desconhecido')
    X_novo_log = pd.get_dummies(X_novo_log, columns=colunas_cat, drop_first=True)
    for col in X_logistic.columns:
        if col not in X_novo_log.columns:
            X_novo_log[col] = 0
    X_novo_log = X_novo_log[X_logistic.columns]
    X_novo_log_scaled = scaler.transform(X_novo_log)
    
    # Fazer predições com todos os modelos
    resultados_classificacao = pd.DataFrame({'ID_Cliente': ids_clientes})
    
    for nome, config in modelos.items():
        modelo = config['modelo']
        X_pred = X_novo_log_scaled if nome == 'Regressão Logística' else X_novo_tree
        pred = modelo.predict(X_pred)
        proba = modelo.predict_proba(X_pred)[:, 1]
        resultados_classificacao[f'{nome}_Pred'] = pred
        resultados_classificacao[f'{nome}_Prob'] = proba
    
    # Votação por maioria
    pred_cols = [c for c in resultados_classificacao.columns if '_Pred' in c]
    resultados_classificacao['Votacao_Maioria'] = (resultados_classificacao[pred_cols].sum(axis=1) >= 2).astype(int)
    resultados_classificacao['Consenso'] = resultados_classificacao[pred_cols].nunique(axis=1) == 1
    
    print("\n=== RESULTADOS CONSOLIDADOS ===")
    display(resultados_classificacao)
    
    print("\n=== RESUMO POR MODELO ===")
    for nome in modelos.keys():
        churn_count = (resultados_classificacao[f'{nome}_Pred'] == 1).sum()
        print(f"  {nome}: {churn_count} clientes em risco de Churn")
    
    print(f"\n  Votação por Maioria: {(resultados_classificacao['Votacao_Maioria'] == 1).sum()} clientes em risco")
    print(f"  Casos com consenso total: {resultados_classificacao['Consenso'].sum()} de {len(resultados_classificacao)}")
    
except FileNotFoundError:
    print(f"Arquivo não encontrado: {ARQUIVO_CLASSIFICAR}")

In [ ]:
# ============================================================
# CONCLUSÃO E RECOMENDAÇÃO
# ============================================================
print("="*70)
print("CONCLUSÃO - COMPARAÇÃO DE MODELOS")
print("="*70)

melhor_modelo = df_resultados.iloc[0]

print(f"\n🏆 MELHOR MODELO: {melhor_modelo['Modelo']}")
print(f"\nMétricas do melhor modelo:")
print(f"  • Acurácia (teste): {melhor_modelo['Acurácia Teste']*100:.2f}%")
print(f"  • AUC-ROC: {melhor_modelo['AUC-ROC']:.4f}")
print(f"  • Precisão: {melhor_modelo['Precisão']*100:.2f}%")
print(f"  • Recall: {melhor_modelo['Recall']*100:.2f}%")
print(f"  • F1-Score: {melhor_modelo['F1-Score']*100:.2f}%")
print(f"  • Validação Cruzada: {melhor_modelo['CV Média']*100:.2f}% (+/- {melhor_modelo['CV Std']*100:.2f}%)")

print("\n" + "-"*70)
print("RANKING COMPLETO (ordenado por AUC-ROC):")
print("-"*70)
for i, row in df_resultados.iterrows():
    print(f"  {i+1}. {row['Modelo']:25s} | AUC: {row['AUC-ROC']:.4f} | Acc: {row['Acurácia Teste']*100:.2f}% | F1: {row['F1-Score']*100:.2f}%")

print("\n" + "="*70)
print("ANÁLISE")
print("="*70)
print("""
CRITÉRIOS DE AVALIAÇÃO:
• AUC-ROC: Capacidade de distinguir entre classes (quanto maior, melhor)
• Acurácia: Proporção de predições corretas
• Precisão: Dos que predisse como Churn, quantos realmente são
• Recall: Dos que realmente são Churn, quantos identificou
• F1-Score: Média harmônica entre Precisão e Recall

CONSIDERAÇÕES PARA NEGÓCIO:
• Se o custo de perder um cliente é alto: priorizar RECALL
• Se campanhas de retenção são caras: priorizar PRECISÃO
• Para equilíbrio geral: considerar F1-Score ou AUC-ROC
""")
print("="*70)